<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.3_Praktikum_Color_Conversion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.3: Praktikum Color Conversion
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengonversi citra BGR ke Grayscale, RGB, HSV, dan CIELAB menggunakan `cv2.cvtColor`.
2. Memahami rentang skala nilai Hue OpenCV ($0 - 179$) dan memecahkan batasan warna merah melingkar.
3. Melakukan segmentasi spektral warna buah sawit matang (oranye/merah) menggunakan `cv2.inRange` dan `cv2.bitwise_and`.
4. Menganalisis ketahanan ruang warna HSV terhadap variasi intensitas pencahayaan lapangan.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Citra Sintetis Buah Sawit Multifraksi Kematangan
Membangkitkan citra 3 buah sawit dengan tingkat kematangan berbeda: Mentah (hitam), Kurang Matang (kuning-hijau), dan Matang (merah-oranye).

In [ ]:
h, w = 350, 600
fruits_bgr = np.full((h, w, 3), 40, dtype=np.uint8)

# Buah 1: Mentah / Unripe (Hitam Keunguan: BGR=(35, 20, 30))
cv2.circle(fruits_bgr, (130, 175), 65, (35, 20, 30), -1)

# Buah 2: Kurang Matang / Underripe (Kuning-Kehijauan: BGR=(20, 180, 200))
cv2.circle(fruits_bgr, (300, 175), 65, (20, 180, 200), -1)

# Buah 3: Matang Sempurna / Ripe (Jingga-Kemerahan: BGR=(15, 75, 230))
cv2.circle(fruits_bgr, (470, 175), 65, (15, 75, 230), -1)

# Konversi Ruang Warna ke HSV dan Grayscale
fruits_gray = cv2.cvtColor(fruits_bgr, cv2.COLOR_BGR2GRAY)
fruits_hsv = cv2.cvtColor(fruits_bgr, cv2.COLOR_BGR2HSV)
fruits_rgb = cv2.cvtColor(fruits_bgr, cv2.COLOR_BGR2RGB)

print(f'Dimensi Citra BGR : {fruits_bgr.shape}')
print(f'Dimensi Grayscale : {fruits_gray.shape}')
print(f'Dimensi HSV       : {fruits_hsv.shape}')


## 2. Segmentasi Buah Matang Berbasis Ambang Rentang HSV
Mengisolasi buah matang berwarna oranye-merah menggunakan `cv2.inRange` dan operasi masking `cv2.bitwise_and`.

In [ ]:
# Rentang warna buah matang (Hue 5 s/d 22, Saturation >= 100, Value >= 100)
lower_ripe = np.array([5, 100, 100], dtype=np.uint8)
upper_ripe = np.array([22, 255, 255], dtype=np.uint8)

mask_ripe = cv2.inRange(fruits_hsv, lower_ripe, upper_ripe)
result_ripe = cv2.bitwise_and(fruits_bgr, fruits_bgr, mask=mask_ripe)

total_fruit_area = np.sum(fruits_gray > 50)
ripe_fruit_area = cv2.countNonZero(mask_ripe)
ripe_percentage = (ripe_fruit_area / max(total_fruit_area, 1)) * 100

print(f'Piksel Area Buah Total  : {total_fruit_area}')
print(f'Piksel Buah Matang      : {ripe_fruit_area}')
print(f'Rasio Kematangan Buah   : {ripe_percentage:.2f}%')


## 3. Visualisasi Dekomposisi Kanal HSV dan Hasil Segmentasi
Menampilkan citra asli, kanal Hue, masker biner, dan hasil isolasi buah matang.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(fruits_rgb)
axes[0].set_title('1. Citra Asli (RGB Display)', fontsize=10, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(fruits_hsv[:, :, 0], cmap='hsv')
axes[1].set_title('2. Kanal Hue (0 - 179)', fontsize=10, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mask_ripe, cmap='gray')
axes[2].set_title('3. Masker Biner Buah Matang', fontsize=10, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(result_ripe, cv2.COLOR_BGR2RGB))
axes[3].set_title(f'4. Buah Matang Terisolasi ({ripe_percentage:.1f}%)', fontsize=10, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.savefig('praktikum_color_segmentation_11_3.png', dpi=150)
plt.show()
